In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [2]:
tr_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

ts_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
])

tr_data = torchvision.datasets.CIFAR100(root='CIFAR', train=True, download=True, transform=tr_transform)
ts_data = torchvision.datasets.CIFAR100(root='CIFAR', train=False, download=True, transform=ts_transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
tr_loader = torch.utils.data.DataLoader(tr_data, batch_size=128, shuffle=True, drop_last= True)
ts_loader = torch.utils.data.DataLoader(ts_data, batch_size=128, shuffle=False, drop_last= True)

In [11]:
class CIFAR(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=2),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2),  
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(num_features=256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size= 2),
            nn.Conv2d(256, 512, kernel_size=3, padding=0),
            nn.BatchNorm2d(num_features=512),
            nn.ReLU(),
            nn.Conv2d(512, 724, kernel_size=3, padding=0),
            nn.BatchNorm2d(num_features=724),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        
        self.dense_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(724,256),
            nn.ReLU(),
            nn.Linear(256, 100)
        )

    
    def forward(self, x):
        x = self.conv_layers(x)
        x = self.dense_layers(x)
        
        return x



In [12]:
lr = 3e-4
num_epochs = 20

In [13]:
torch.manual_seed(2009)

model = CIFAR().to("cuda")
#model.load_state_dict(torch.load("model_cifar.pth"))

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

In [15]:
model.train()
for epoch in range(num_epochs):
    n_c = 0
    for batch_num, (X,y) in enumerate(tr_loader):
        X = X.to("cuda")
        y = y.to("cuda")
        logits = model(X)
        
        loss = loss_fn(logits,y)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        n_c += (y == torch.argmax(torch.softmax(model(X), dim = 1), dim = 1)).sum().item()

      #  if batch_num % 30 == 0:
          #  torch.save({'model_state_dict': model.state_dict()}, "model_cifar.pth")
    
    print(f"Epoch : {epoch} Accuracy : {n_c / (len(y) * (batch_num + 1))}")

Epoch : 0 Accuracy : 0.2462940705128205
Epoch : 1 Accuracy : 0.4117588141025641
Epoch : 2 Accuracy : 0.5064503205128205
Epoch : 3 Accuracy : 0.568790064102564
Epoch : 4 Accuracy : 0.6163461538461539
Epoch : 5 Accuracy : 0.6575320512820513
Epoch : 6 Accuracy : 0.693369391025641
Epoch : 7 Accuracy : 0.72734375
Epoch : 8 Accuracy : 0.7584334935897435
Epoch : 9 Accuracy : 0.7897636217948718
Epoch : 10 Accuracy : 0.8186698717948718
Epoch : 11 Accuracy : 0.8450921474358974
Epoch : 12 Accuracy : 0.8702123397435897
Epoch : 13 Accuracy : 0.8915464743589744
Epoch : 14 Accuracy : 0.9120392628205128
Epoch : 15 Accuracy : 0.9318108974358974
Epoch : 16 Accuracy : 0.9442107371794872
Epoch : 17 Accuracy : 0.9550480769230769


KeyboardInterrupt: 

In [16]:
model.eval()

n_c = 0
for batch_num, (X,y) in enumerate(ts_loader):
    X = X.to("cuda")
    y = y.to("cuda")
    logits = model(X)

    n_c += (y == torch.argmax(torch.softmax(model(X), dim = 1), dim = 1)).sum().item()

print(f"Accuracy : {n_c / (len(y) * (batch_num + 1))}")

Accuracy : 0.6159855769230769


In [19]:
model(X).shape

torch.Size([128, 100])

In [23]:
y.shape

torch.Size([128])

In [31]:
(y == torch.argmax(torch.softmax(model(X), dim = 1), dim = 0))

RuntimeError: The size of tensor a (128) must match the size of tensor b (100) at non-singleton dimension 0

In [35]:
torch.argmax(torch.softmax(model(X), dim = 1), dim = 1)

tensor([92, 58, 42, 58, 53, 17, 44, 53, 92, 46, 44, 23, 61, 94, 58, 93, 42, 70,
        70, 24, 44, 44, 58, 53, 93, 42, 53, 53, 93, 17, 70, 24, 44, 53, 44, 61,
        73, 44, 21, 44, 53, 53, 53, 42, 58, 44,  2,  2,  6, 44, 17, 33, 73, 58,
        58, 94, 44, 43, 94, 33, 74, 61, 92, 44, 17, 58, 44, 44, 44, 61, 44, 53,
        24, 24, 58, 43,  2, 24, 23, 44, 44, 53, 71, 43, 93, 58, 53, 61, 93, 23,
        61, 44, 44, 24, 70, 53, 70, 92, 44, 23, 17, 44, 92, 24, 61, 43, 17, 17,
        77, 42, 61, 17, 17, 44, 23, 53, 61, 58, 24, 44, 75, 65, 44, 92, 61, 61,
        93, 53])